In [1]:
!pip install langchain langchain-community langchain-openai \
             faiss-cpu pypdf sentence-transformers openai tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependen

In [12]:
# Step 1: install the separate package (one-time)
!pip install -q langchain-text-splitters

In [4]:
import requests
import os

# Mimic a real browser so servers don't block the request
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

docs_to_download = {
    # NIST AI Risk Management Framework (AI RMF 1.0)
    "nist_ai_rmf.pdf":
        "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf",

    # NIST Generative AI Profile (AI 600-1) - newer companion doc
    "nist_genai_profile.pdf":
        "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.600-1.pdf",

    # CFPB Supervision & Examination Manual
    "cfpb_manual.pdf":
        "https://files.consumerfinance.gov/f/documents/cfpb_supervision-and-examination-manual.pdf",
}

os.makedirs("docs", exist_ok=True)

for filename, url in docs_to_download.items():
    path = f"docs/{filename}"
    if os.path.exists(path):
        print(f"✓ Already exists: {filename}")
        continue
    print(f"Downloading {filename}...")
    try:
        resp = requests.get(url, headers=HEADERS, timeout=60, stream=True)
        resp.raise_for_status()
        with open(path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"  ✓ Saved ({os.path.getsize(path) // 1024} KB)")
    except Exception as e:
        print(f"  ✗ Failed: {e}")

  ✓ Saved (1900 KB)
  ✓ Saved (1147 KB)
  ✗ Failed: 403 Client Error: Forbidden for url: https://files.consumerfinance.gov/f/documents/cfpb_supervision-and-examination-manual.pdf


In [7]:
!wget -q --user-agent="Chrome/122.0.0.0 Safari/537.36" \
  "https://files.consumerfinance.gov/f/documents/cfpb_supervision-and-examination-manual.pdf" \
  -O docs/cfpb_manual.pdf && echo "✓ cfpb_manual.pdf"

In [9]:
!ls -la docs/


total 3060
drwxr-xr-x 2 root root    4096 Apr 23 06:46 .
drwxr-xr-x 1 root root    4096 Apr 23 06:43 ..
-rw-r--r-- 1 root root       0 Apr 23 06:47 cfpb_manual.pdf
-rw-r--r-- 1 root root 1946127 Apr 23 06:45 nist_ai_rmf.pdf
-rw-r--r-- 1 root root 1174643 Apr 23 06:45 nist_genai_profile.pdf


## Load and Parse the PDF now

In [10]:
from langchain_community.document_loaders import PyPDFLoader

all_docs = []

for filename in os.listdir("docs"):
    if filename.endswith(".pdf"):
        loader = PyPDFLoader(f"docs/{filename}")
        pages = loader.load()
        # tag each page with source metadata
        for page in pages:
            page.metadata["source_file"] = filename
        all_docs.extend(pages)
        print(f"{filename}: {len(pages)} pages loaded")

print(f"\nTotal pages loaded: {len(all_docs)}")

nist_genai_profile.pdf: 64 pages loaded
nist_ai_rmf.pdf: 48 pages loaded

Total pages loaded: 112


## Chunk the documents

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(all_docs)

print(f"Total chunks created: {len(chunks)}")
print(f"\nSample chunk:\n{chunks[10].page_content}")
print(f"\nMetadata: {chunks[10].metadata}")

Total chunks created: 637

Sample chunk:
1 
1. Introduction 
This document is a cross-sectoral proﬁle of and companion resource for the AI Risk Management 
Framework (AI RMF 1.0) for Generative AI,1 pursuant to President Biden’s Executive Order (EO) 14110 on 
Safe, Secure, and Trustworthy Artiﬁcial Intelligence.2 The AI RMF was released in January 2023, and is 
intended for voluntary use and to improve the ability of organizations to incorporate trustworthiness

Metadata: {'producer': 'Adobe PDF Library 24.2.159', 'creator': 'Acrobat PDFMaker 24 for Word', 'creationdate': '2024-08-05T14:17:02-04:00', 'author': 'National Institute of Standards and Technology', 'category': 'NIST AI 600-1', 'comments': '', 'company': '', 'contenttypeid': '0x01010068CEA9BE6E0AF749888425167690E526', 'keywords': '', 'mediaserviceimagetags': '', 'moddate': '2025-03-24T15:11:27-04:00', 'sourcemodified': 'D:20240805160221', 'subject': '', 'title': 'Artificial Intelligence Risk Management Framework: Generative A

##Embed & store in FAISS (free, no API key needed)

In [15]:
# Using a free HuggingFace embedding model — no OpenAI key needed
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Loading embedding model (downloads once ~90MB)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Building FAISS index — this may take a minute...")
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save to disk so you don't re-embed every run
vectorstore.save_local("faiss_legal_index")
print("Vector store saved to ./faiss_legal_index")

Loading embedding model (downloads once ~90MB)...


/tmp/ipykernel_7117/4253992188.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS index — this may take a minute...
Vector store saved to ./faiss_legal_index


##Test a retrieval query

In [16]:
# Load from disk (useful for subsequent runs)
vectorstore = FAISS.load_local(
    "faiss_legal_index",
    embeddings,
    allow_dangerous_deserialization=True
)

query = "What are the obligations of AI system providers under the EU AI Act?"
results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source_file')} | Page: {doc.metadata.get('page')}")
    print(doc.page_content[:400])


--- Result 1 ---
Source: nist_ai_rmf.pdf | Page: 40
authority for acquisition of AI models, products, or services from a third-party developer,
vendor, or contractor.
Governance and Oversight tasks are assumed by AI actors with management, fiduciary,
and legal authority and responsibility for the organization in which an AI system is de-
Page 36

--- Result 2 ---
Source: nist_ai_rmf.pdf | Page: 39
design), governance experts, data engineers, data providers, system funders, product man-
agers, third-party entities, evaluators, and legal and privacy governance.
AI Development tasks are performed during the AI Model phase of the lifecycle in Figure
2. AI Development actors provide the initial infrastructure of AI systems and are responsi-
ble for model building and interpretation tasks, which 

--- Result 3 ---
Source: nist_genai_profile.pdf | Page: 16
those related to data privacy, copyright and intellectual property law. 
Data Privacy; Harmful Bias and 
Homogenization; Intellectual 
Pr